# Test inference

Attach the soil competition data and soil-cv-utils version 10.
Turn Internet on and allow WANDB_API_KEY for this notebook.
Each group uses all four folds of one recipe. One group makes one CSV.
There is no training or new W&B run.


In [ ]:
from pathlib import Path

CHECKPOINTS = [
    {
        "name": "mil-r18-cosine-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/mil-r18-cosine-fold-0:v0",
            "public-team/soil-grain-size/mil-r18-cosine-fold-1:v0",
            "public-team/soil-grain-size/mil-r18-cosine-fold-2:v0",
            "public-team/soil-grain-size/mil-r18-cosine-fold-3:v0",
        ],
    },
    {
        "name": "mil-r18-288-4fold",
        "checkpoints": [
            "public-team/soil-grain-size/mil-r18-288-fold-0:v0",
            "public-team/soil-grain-size/mil-r18-288-fold-1:v0",
            "public-team/soil-grain-size/mil-r18-288-fold-2:v0",
            "public-team/soil-grain-size/mil-r18-288-fold-3:v0",
        ],
    },
    None,
]
DEVICE = "auto"
OUTPUT_DIR = Path("/kaggle/working/submissions")


## Load the code

A single run ID, run URL, or exact artifact can also be used as one list item.
A group must have folds 0, 1, 2, and 3 from the same recipe and study.
The source and input settings are checked when we load each model.


In [ ]:
import os
import sys

import torch
import wandb
from kaggle_secrets import UserSecretsClient

code_root = Path(os.environ.get(
    "SOILCV_INFERENCE_CODE_ROOT",
    "/kaggle/input/datasets/denysmykhailov/soil-cv-utils",
))
if not (code_root / "soilcv/inference/runner.py").is_file():
    matches = list(Path("/kaggle/input").rglob("soilcv/inference/runner.py"))
    if len(matches) != 1:
        raise RuntimeError("Attach the current soil-cv-utils dataset.")
    code_root = matches[0].parents[2]
sys.path.insert(0, str(code_root))
from soilcv.inference.runner import run_checkpoints

try:
    key = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as error:
    raise RuntimeError("Enable WANDB_API_KEY in Add-ons > Secrets, then Run All.") from error
os.environ["WANDB_API_KEY"] = key
wandb.login(key=key, verify=True)
del key
print("GPU available:", torch.cuda.is_available())


## Predict the whole test set

Each model uses its saved input settings. All photos are used.
We average the four curves for each test sample.
If one fold is not ready, the whole group is skipped.
Rows follow sample_submission.csv. IDs and valid curves are checked.


In [ ]:
results = run_checkpoints(CHECKPOINTS, OUTPUT_DIR, device=DEVICE)
print("Saved CSV files:", len(results))


## Ready submissions

One CSV belongs to one model or one complete four-fold group.
inference_manifest.json records all exact model versions and source runs.
The notebook saves files. It does not submit them to the competition.


In [ ]:
import pandas as pd
from IPython.display import FileLink, display

assert results, "No ready predictions yet."
for result in results:
    path = Path(result["output"])
    frame = pd.read_csv(path)
    print(path.name, "| shape:", frame.shape)
    display(FileLink(str(path)))
    display(frame.head(2))
display(pd.DataFrame(results)[["slot", "name", "ensemble", "device", "output"]])
print("Outputs:", OUTPUT_DIR)
